# Day 1 — Number Theory and Modular Arithmetic
**Student:** Md. Shishir Kaysar Shuvon  
**Student ID:** `IT-23030`  
**Department:** ICT, MBSTU

## Learning objectives
This notebook follows the Day 1 lecture: GCD and Euclid's algorithm, LCM, linear Diophantine equations, modular division and inverses, Chinese Remainder Theorem (CRT), fast modular exponentiation, Fermat's Little Theorem, Euler's totient function and Euler's theorem.

> Run every code cell in order. The examples are small enough to verify by hand.

## 1. Greatest Common Divisor (GCD)

The greatest common divisor of integers `a` and `b` is the largest positive integer that divides both. Euclid's algorithm uses

\[
\gcd(a,b)=\gcd(b,a\bmod b).
\]

Repeat until the second value becomes zero. The remaining first value is the GCD.

In [ ]:
def gcd(a, b):
    a, b = abs(a), abs(b)
    while b != 0:
        a, b = b, a % b
    return a

print("gcd(24, 16) =", gcd(24, 16))
print("gcd(1071, 462) =", gcd(1071, 462))

**Expected output**
```text
gcd(24, 16) = 8
gcd(1071, 462) = 21
```
**Complexity:** Euclid's algorithm takes `O(log(min(|a|, |b|)))` divisions in the standard analysis and uses `O(1)` auxiliary space.

## 2. Least Common Multiple (LCM)

For positive integers,

\[
\operatorname{lcm}(a,b)\cdot\gcd(a,b)=a\cdot b,
\qquad
\operatorname{lcm}(a,b)=\frac{|ab|}{\gcd(a,b)}.
\]

In [ ]:
def lcm(a, b):
    if a == 0 or b == 0:
        return 0
    return abs(a // gcd(a, b) * b)

print("lcm(24, 16) =", lcm(24, 16))
print("Check:", lcm(24, 16) * gcd(24, 16) == 24 * 16)

## 3. Extended Euclidean Algorithm

The extended algorithm finds integers `x` and `y` such that

\[
ax+by=\gcd(a,b).
\]

These coefficients are called Bézout coefficients. They are useful for solving Diophantine equations and finding modular inverses.

In [ ]:
def extended_gcd(a, b):
    if b == 0:
        return abs(a), 1 if a >= 0 else -1, 0
    g, x1, y1 = extended_gcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return g, x, y

g, x, y = extended_gcd(187, 55)
print(f"gcd = {g}; x = {x}; y = {y}")
print("Verification:", 187*x + 55*y)

## 4. Linear Diophantine Equations

A linear Diophantine equation has the form

\[
ax+by=c,\qquad x,y\in\mathbb Z.
\]

It has an integer solution **if and only if** `gcd(a,b)` divides `c`.

If `(x0,y0)` is one solution, let `d=gcd(a,b)`, `p=a/d`, and `q=b/d`. All integer solutions are

\[
x=x_0+tq,\qquad y=y_0-tp,\qquad t\in\mathbb Z.
\]

In [ ]:
def solve_diophantine(a, b, c):
    if a == 0 and b == 0:
        return (0, 0) if c == 0 else None
    g, x, y = extended_gcd(a, b)
    if c % g != 0:
        return None
    k = c // g
    return x*k, y*k

a, b, c = 187, 55, 121
solution = solve_diophantine(a, b, c)
print("One solution:", solution)
if solution:
    x0, y0 = solution
    d = gcd(a, b)
    print("Check:", a*x0 + b*y0)
    print("General solution: x =", x0, "+ t*", b//d,
          "; y =", y0, "- t*", a//d)
print("No-solution example:", solve_diophantine(187, 55, 45))

## 5. Modular Arithmetic and Modular Inverse

`a ≡ b (mod n)` means `n` divides `a-b`. A multiplicative inverse of `a` modulo `n` is a number `u` satisfying

\[
au\equiv 1\pmod n.
\]

An inverse exists **if and only if** `gcd(a,n)=1`. Modular division is multiplication by this inverse.

In [ ]:
def modular_inverse(a, n):
    g, x, _ = extended_gcd(a, n)
    if g != 1:
        return None
    return x % n

a, b, n = 2, 7, 9
inv = modular_inverse(a, n)
print("Inverse of 2 modulo 9:", inv)
print("7 / 2 modulo 9 =", (b * inv) % n)
print("Verification:", (2 * inv) % 9)

**Lecture example:** `gcd(9,2)=1`; extended Euclid gives `9×1 + 2×(-4)=1`. Thus `-4 ≡ 5 (mod 9)` is the inverse of 2, and `7/2 ≡ 7×5 ≡ 8 (mod 9)`.

If `gcd(a,n) != 1`, division by `a` modulo `n` is not generally defined as a unique inverse.

## 6. Chinese Remainder Theorem (CRT)

For pairwise coprime positive moduli `n1,...,nk`, the system

\[
x\equiv a_i\pmod{n_i}
\]

has a unique solution modulo `N=n1n2...nk`. For two coprime moduli, CRT combines the two remainders into one remainder modulo their product.

In [ ]:
def chinese_remainder_two(a1, n1, a2, n2):
    if gcd(n1, n2) != 1:
        raise ValueError("The two moduli must be coprime.")
    inv = modular_inverse(n1, n2)
    k = ((a2 - a1) * inv) % n2
    x = a1 + n1*k
    return x % (n1*n2)

x = chinese_remainder_two(2, 3, 3, 5)
print("x =", x)
print("Check:", x % 3, x % 5)

## 7. Fast Modular Exponentiation

We want to compute `b^e mod m` without constructing the potentially huge integer `b^e`. Repeated squaring reduces the exponent by roughly half at each step.

\[
b^e\bmod m
\]

The number of modular multiplications is `O(log e)`.

In [ ]:
def mod_pow(base, exponent, modulus):
    if modulus <= 0:
        raise ValueError("Modulus must be positive.")
    if exponent < 0:
        raise ValueError("This implementation expects a non-negative exponent.")
    result = 1
    base %= modulus
    while exponent > 0:
        if exponent & 1:
            result = (result * base) % modulus
        base = (base * base) % modulus
        exponent >>= 1
    return result

print("7^4 mod 11 =", mod_pow(7, 4, 11))
print("7^13 mod 11 =", mod_pow(7, 13, 11))
print("Cross-check:", pow(7, 13, 11))

## 8. Fermat's Little Theorem

If `p` is prime and `p` does not divide `a`, then

\[
a^{p-1}\equiv1\pmod p.
\]

Equivalently, for every integer `a`, `a^p ≡ a (mod p)`. This theorem is useful in modular arithmetic and primality tests, but its converse is false: composite numbers can pass a Fermat test for some bases.

In [ ]:
def fermat_check(a, p):
    return pow(a, p-1, p) == 1 if gcd(a, p) == 1 else None

print("For p=7, a=3:", fermat_check(3, 7))
print("3^6 mod 7 =", pow(3, 6, 7))

## 9. Euler's Totient Function

Euler's totient `φ(n)` counts the integers from 1 through `n` that are coprime to `n` (equivalently, the lecture describes the coprime residues modulo `n`; note the special convention `φ(1)=1`).

For a prime `p`, `φ(p)=p-1`. If the prime factorization is known,

\[
n=\prod_i p_i^{a_i}
\quad\Longrightarrow\quad
\phi(n)=n\prod_{p\mid n}\left(1-\frac1p\right).
\]

In [ ]:
def euler_phi(n):
    if n < 1:
        raise ValueError("n must be positive")
    result = n
    x = n
    p = 2
    while p*p <= x:
        if x % p == 0:
            while x % p == 0:
                x //= p
            result -= result // p
        p += 1
    if x > 1:
        result -= result // x
    return result

for n in [1, 2, 3, 10, 12, 17]:
    print(f"phi({n}) = {euler_phi(n)}")

## 10. Euler's Theorem

If `gcd(a,n)=1`, then

\[
a^{\phi(n)}\equiv1\pmod n.
\]

It generalizes Fermat's Little Theorem. Together with modular exponentiation, it is relevant to public-key cryptography such as RSA. This notebook demonstrates the number-theory identity only; it is not a secure RSA implementation.

In [ ]:
a, n = 3, 10
print("gcd(a,n) =", gcd(a, n))
print("a^phi(n) mod n =", pow(a, euler_phi(n), n))

## Summary and observations

- Euclid's algorithm computes GCD efficiently.
- LCM follows from `lcm(a,b)=|ab|/gcd(a,b)`.
- Extended Euclid returns Bézout coefficients and supports modular inverses.
- `ax+by=c` is solvable over integers exactly when `gcd(a,b)` divides `c`.
- CRT combines congruences with pairwise coprime moduli.
- Repeated squaring computes modular powers in logarithmic time.
- Fermat's and Euler's theorems provide useful modular identities.

## References
- Course resource: **Day 1.pdf**, supplied with the assignment.